# From a single neuron to ResNet — build a CIFAR-10 image classifier

_Capstones_

**Walk the path from the backprop rule to a small ResNet, one landmark paper at a time, then assemble and train the working classifier.**

---

This notebook is a practice scaffold for the **From a single neuron to ResNet — build a CIFAR-10 image classifier** lesson. We build the network in small, labelled steps — the data tensors, the residual block, a stack of stages, the training loop — and finish by running the skip-connection ablation that shows *why* ResNet works.

Cells marked **🎛️ Play with it** are interactive sandboxes: in Colab they show sliders and dropdowns, and outside Colab they fall back to a default run. Run each cell top to bottom, then change the values to build intuition. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / matplotlib ship with Colab.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. CIFAR-10 images are tiny `32 × 32` color tensors; the label is one of ten object classes. We inspect raw pixels first, then the normalized tensors that the network will see.

### Step 1 — Load CIFAR-10 and peek at raw images

The preview dataset returns PIL images, so we can see the pictures before any normalization. The model will later consume transformed tensors, but this raw grid answers the first human question: *what are we asking the classifier to recognize?*

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"

preview = torchvision.datasets.CIFAR10(root="./data", train=True, download=True)
classes = preview.classes
print("dataset: CIFAR10")
print("train samples:", len(preview), " image size:", preview[0][0].size, " classes:", len(classes))
print("classes:", classes)

fig, axes = plt.subplots(2, 5, figsize=(9, 4))
for ax, idx in zip(axes.ravel(), range(10)):
    image, label = preview[idx]
    ax.imshow(image)
    ax.set_title(classes[label])
    ax.axis("off")
plt.suptitle("CIFAR-10 samples")
plt.tight_layout()
plt.show()

### Step 2 — Labels as a small table

Neural networks never see the class names directly; they see integer ids. This table keeps a few examples concrete: index, label id, human-readable class, and image shape.

In [ ]:
rows = []
for idx in range(8):
    image, label = preview[idx]
    rows.append({"index": idx, "label id": label, "class": classes[label], "PIL size": image.size})
label_df = pd.DataFrame(rows)
display(label_df)

### Step 3 — Class balance in the subset we train on

The full CIFAR-10 train split is balanced. We deliberately train on the first `8000` images to keep the capstone fast, so we check the subset balance too. A fair ablation needs both models to see exactly the same data.

In [ ]:
TRAIN_N = 8000
TEST_N = 2000
BATCH_SIZE = 128
TEST_BATCH_SIZE = 256
EPOCHS = 8

train_labels = np.array(preview.targets[:TRAIN_N])
counts = np.bincount(train_labels, minlength=10)
balance_df = pd.DataFrame({"class": classes, "count in train subset": counts})
display(balance_df)

plt.figure(figsize=(8, 3))
plt.bar(classes, counts, color="#79c0ff")
plt.xticks(rotation=35, ha="right")
plt.ylabel("count")
plt.title("class counts")
plt.tight_layout()
plt.show()

### Step 4 — Build the normalized tensors

`ToTensor()` changes each image from PIL format into a tensor with shape `(channels, height, width) = (3, 32, 32)`. `Normalize(mean, std)` then centers each color channel using the standard CIFAR-10 statistics. The network sees these normalized values, not raw `0..255` pixels.

In [ ]:
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2470, 0.2430, 0.2610)

tfm = T.Compose([
    T.ToTensor(),
    T.Normalize(CIFAR_MEAN, CIFAR_STD),
])
raw_to_tensor = T.ToTensor()
train_full = torchvision.datasets.CIFAR10(root="./data", train=True,  download=True, transform=tfm)
test_full  = torchvision.datasets.CIFAR10(root="./data", train=False, download=True, transform=tfm)
train_set  = torch.utils.data.Subset(train_full, range(TRAIN_N))   # small + fast
test_set   = torch.utils.data.Subset(test_full,  range(TEST_N))   # held-out images
train_loader = torch.utils.data.DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = torch.utils.data.DataLoader(test_set,  batch_size=TEST_BATCH_SIZE)

x0, y0 = train_set[0]
print("one transformed image tensor:", tuple(x0.shape), x0.dtype, " label:", y0, classes[y0])
print("min/max after normalize:", round(float(x0.min()), 3), round(float(x0.max()), 3))

### Step 5 — Per-channel normalization statistics

Here we compute raw channel means/stds on a small sample and compare them with the constants used for normalization. After normalization, the batch is roughly centered near zero with similar scale in each color channel.

In [ ]:
stat_n = min(1024, len(preview))
raw_batch = torch.stack([raw_to_tensor(preview[i][0]) for i in range(stat_n)])
norm_batch = torch.stack([train_full[i][0] for i in range(stat_n)])
stats_df = pd.DataFrame({
    "channel": ["red", "green", "blue"],
    "raw mean": raw_batch.mean(dim=(0, 2, 3)).numpy(),
    "raw std": raw_batch.std(dim=(0, 2, 3)).numpy(),
    "normalize mean": CIFAR_MEAN,
    "normalize std": CIFAR_STD,
    "after norm mean": norm_batch.mean(dim=(0, 2, 3)).numpy(),
    "after norm std": norm_batch.std(dim=(0, 2, 3)).numpy(),
})
display(stats_df.round(3))

plt.figure(figsize=(7, 3))
plt.hist(norm_batch[:, 0].flatten().numpy(), bins=40, alpha=0.65, label="red")
plt.hist(norm_batch[:, 1].flatten().numpy(), bins=40, alpha=0.45, label="green")
plt.hist(norm_batch[:, 2].flatten().numpy(), bins=40, alpha=0.45, label="blue")
plt.title("normalized pixels")
plt.xlabel("value")
plt.ylabel("count")
plt.legend()
plt.tight_layout()
plt.show()

### Step 6 — A real mini-batch

Training happens in batches. A batch has shape `(B, C, H, W)`; for CIFAR-10 that means many `3 × 32 × 32` images stacked together. We also define a small `denorm` helper so plots can turn normalized tensors back into viewable images.

In [ ]:
def denorm(x):
    mean = torch.tensor(CIFAR_MEAN, device=x.device).view(3, 1, 1)
    std = torch.tensor(CIFAR_STD, device=x.device).view(3, 1, 1)
    return (x * std + mean).clamp(0, 1)

xb_data, yb_data = next(iter(train_loader))
print("image batch:", tuple(xb_data.shape))
print("label batch:", tuple(yb_data.shape), yb_data[:10].tolist())

fig, axes = plt.subplots(2, 6, figsize=(9, 3.4))
for ax, image, label in zip(axes.ravel(), xb_data[:12], yb_data[:12]):
    ax.imshow(denorm(image).permute(1, 2, 0).numpy())
    ax.set_title(classes[int(label)])
    ax.axis("off")
plt.suptitle("training batch")
plt.tight_layout()
plt.show()

## Reference implementation — PyTorch

Each line in the final classifier corresponds to one landmark idea:

| Paper / idea | What it contributes | In our code |
|---|---|---|
| Backpropagation | gradients for every layer | `loss.backward()` |
| LeNet | weight-shared image convolutions | `nn.Conv2d(...)` |
| AlexNet | ReLU nonlinearity and deep CNN practice | `nn.ReLU(...)` |
| BatchNorm | stable channel-wise normalization | `nn.BatchNorm2d(...)` |
| VGG | stacks of small `3×3` filters | two `3×3` convs per block |
| ResNet | identity skip connections | `out + identity` |
| Adam | adaptive optimizer | `torch.optim.Adam(...)` |

The heart of this notebook is the **residual block**, built tensor by tensor on one real image batch before we package it as `BasicBlock`.

### Step 7 — Hero batch: the input image

We will follow one batch through one residual block. The input tensor has 16 feature channels because a small stem convolution has already converted RGB pixels into low-level edges and colors.

In [ ]:
hero_images, hero_labels = next(iter(train_loader))
hero_images = hero_images[:8]
hero_labels = hero_labels[:8]

hero_stem = nn.Sequential(
    nn.Conv2d(3, 16, 3, padding=1, bias=False),
    nn.BatchNorm2d(16),
    nn.ReLU(inplace=False),
)
hero_block_layers = nn.ModuleDict({
    "conv1": nn.Conv2d(16, 16, 3, stride=1, padding=1, bias=False),
    "bn1": nn.BatchNorm2d(16),
    "conv2": nn.Conv2d(16, 16, 3, stride=1, padding=1, bias=False),
    "bn2": nn.BatchNorm2d(16),
})
hero_stem.eval(); hero_block_layers.eval()

with torch.no_grad():
    hero_x = hero_stem(hero_images)

print("raw image batch:", tuple(hero_images.shape))
print("after stem:", tuple(hero_x.shape))
plt.figure(figsize=(2.5, 2.5))
plt.imshow(denorm(hero_images[0]).permute(1, 2, 0).numpy())
plt.title(classes[int(hero_labels[0])])
plt.axis("off")
plt.show()

### Step 8 — `conv1 → BatchNorm → ReLU`

The first half of the block maps `16` channels to `16` channels with a `3×3` convolution, normalizes each channel, and clips negative values with ReLU. The spatial shape stays `32×32` because padding is `1`.

In [ ]:
with torch.no_grad():
    h_conv1 = hero_block_layers["conv1"](hero_x)
    h_bn1 = hero_block_layers["bn1"](h_conv1)
    h_relu1 = F.relu(h_bn1)

shape_df = pd.DataFrame([
    {"tensor": "x", "shape": tuple(hero_x.shape), "mean": float(hero_x.mean()), "std": float(hero_x.std())},
    {"tensor": "conv1(x)", "shape": tuple(h_conv1.shape), "mean": float(h_conv1.mean()), "std": float(h_conv1.std())},
    {"tensor": "BN(conv1)", "shape": tuple(h_bn1.shape), "mean": float(h_bn1.mean()), "std": float(h_bn1.std())},
    {"tensor": "ReLU", "shape": tuple(h_relu1.shape), "mean": float(h_relu1.mean()), "std": float(h_relu1.std())},
])
display(shape_df.round(3))

fig, axes = plt.subplots(1, 6, figsize=(9, 1.8))
for ax, ch in zip(axes, range(6)):
    ax.imshow(h_relu1[0, ch].numpy(), cmap="magma")
    ax.set_title(f"ch {ch}")
    ax.axis("off")
plt.suptitle("after conv1 bn relu")
plt.tight_layout()
plt.show()

### Step 9 — `conv2 → BatchNorm` gives `F(x)`

The second convolution produces the residual function `F(x)`. Notice there is **no ReLU yet**: ResNet applies the final ReLU *after* adding the identity path.

In [ ]:
with torch.no_grad():
    h_conv2 = hero_block_layers["conv2"](h_relu1)
    Fx = hero_block_layers["bn2"](h_conv2)

fx_df = pd.DataFrame([
    {"tensor": "ReLU1", "shape": tuple(h_relu1.shape), "mean": float(h_relu1.mean()), "std": float(h_relu1.std())},
    {"tensor": "conv2", "shape": tuple(h_conv2.shape), "mean": float(h_conv2.mean()), "std": float(h_conv2.std())},
    {"tensor": "F(x)", "shape": tuple(Fx.shape), "mean": float(Fx.mean()), "std": float(Fx.std())},
])
display(fx_df.round(3))

fig, axes = plt.subplots(1, 6, figsize=(9, 1.8))
for ax, ch in zip(axes, range(6)):
    ax.imshow(Fx[0, ch].numpy(), cmap="coolwarm")
    ax.set_title(f"F ch {ch}")
    ax.axis("off")
plt.suptitle("residual F(x)")
plt.tight_layout()
plt.show()

### Step 10 — Add the identity: `F(x) + x`

This is the ResNet move. If learning `F(x)` is hard, the block can still pass information and gradients through `x`. The output keeps exactly the same `(channels, height, width)` shape as the input.

In [ ]:
with torch.no_grad():
    with_skip = F.relu(Fx + hero_x)
    no_skip = F.relu(Fx)

add_df = pd.DataFrame([
    {"path": "identity x", "shape": tuple(hero_x.shape), "mean": float(hero_x.mean()), "std": float(hero_x.std())},
    {"path": "residual F(x)", "shape": tuple(Fx.shape), "mean": float(Fx.mean()), "std": float(Fx.std())},
    {"path": "ReLU(F(x)+x)", "shape": tuple(with_skip.shape), "mean": float(with_skip.mean()), "std": float(with_skip.std())},
    {"path": "ReLU(F(x))", "shape": tuple(no_skip.shape), "mean": float(no_skip.mean()), "std": float(no_skip.std())},
])
display(add_df.round(3))

fig, axes = plt.subplots(1, 3, figsize=(8, 2.4))
for ax, tensor, title in zip(axes, [hero_x, Fx, with_skip], ["x", "F(x)", "F(x)+x"]):
    ax.imshow(tensor[0, 0].numpy(), cmap="viridis")
    ax.set_title(title)
    ax.axis("off")
plt.suptitle("skip add")
plt.tight_layout()
plt.show()

**🎛️ Play with it — residual skip on/off**

Toggle the skip connection inside this one block. **Try:** `skip=True`, then `False`. **Watch:** output statistics and the note about gradient flow. **Why it matters:** the ablation later changes only this switch across the whole network.

In [ ]:
def play(skip=True):
    out = F.relu(Fx + hero_x) if skip else F.relu(Fx)
    stats = pd.DataFrame({
        "setting": ["skip on" if skip else "skip off"],
        "shape": [tuple(out.shape)],
        "mean": [float(out.mean())],
        "std": [float(out.std())],
        "zero fraction": [float((out == 0).float().mean())],
    })
    display(stats.round(3))
    print("gradient note:", "identity path gives a direct route backward" if skip else "gradients must pass through both conv layers")
    plt.figure(figsize=(5, 2.8))
    plt.hist(out.flatten().numpy(), bins=40, color="#7ee787")
    plt.title("block output stats")
    plt.xlabel("activation")
    plt.ylabel("count")
    plt.show()

try:
    from ipywidgets import interact, Checkbox
    interact(play, skip=Checkbox(value=True, description="skip"))
except Exception:
    play()

### Step 11 — Package the residual block as `BasicBlock`

The hand-built path above becomes a reusable module. The `skip` flag is the ablation switch: with it on we add the identity (a real ResNet block); with it off we get a matched plain block of the same depth. The projection shortcut appears only when channel count or spatial size changes.

In [ ]:
# The residual block: conv -> BN -> ReLU -> conv -> BN, then + x, then ReLU.
#   skip=True  -> ResNet (the residual block, ResNet 2015).
#   skip=False -> the ABLATION: a matched plain net (same depth, no "+ identity").
# Pieces by paper: nn.Conv2d (LeNet, 3x3 per VGG), nn.BatchNorm2d (BatchNorm),
#                  nn.ReLU (AlexNet), out + identity (ResNet).
class BasicBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1, skip=True):
        super().__init__()
        self.skip = skip
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)
        self.relu = nn.ReLU(inplace=True)

        # Projection shortcut (ResNet Eqn. 2, W_s): only when channels or stride change.
        self.proj = None
        if stride != 1 or in_ch != out_ch:
            self.proj = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_ch))

    def forward(self, x):
        identity = x
        out = self.relu(self.bn1(self.conv1(x)))   # conv -> BN -> ReLU
        out = self.bn2(self.conv2(out))            # conv -> BN  (this is F(x))
        if self.skip:                              # the ablation switch
            if self.proj is not None:
                identity = self.proj(x)            # W_s x  when dims differ
            out = out + identity                   # ResNet Eqn. 1: F(x) + x
        return self.relu(out)                      # ReLU AFTER the add

### Step 12 — Inspect block shapes and parameters

A same-channel block preserves `16×32×32`. A downsampling block uses stride `2`, doubles channels, and needs a projection shortcut so the identity can be added to `F(x)`.

In [ ]:
def count_params(module):
    return sum(p.numel() for p in module.parameters() if p.requires_grad)

same_block = BasicBlock(16, 16, stride=1, skip=True)
down_block = BasicBlock(16, 32, stride=2, skip=True)
with torch.no_grad():
    same_out = same_block(hero_x.clone())
    down_out = down_block(hero_x.clone())
block_df = pd.DataFrame([
    {"block": "same shape", "input": tuple(hero_x.shape), "output": tuple(same_out.shape), "projection": same_block.proj is not None, "params": count_params(same_block)},
    {"block": "downsample", "input": tuple(hero_x.shape), "output": tuple(down_out.shape), "projection": down_block.proj is not None, "params": count_params(down_block)},
])
display(block_df)

**🎛️ Play with it — convolution output size**

Change stride and padding for a `3×3` convolution on a `32×32` image. **Try:** stride `1` vs `2`, padding `0` vs `1`. **Watch:** the formula predict the output size. **Why it matters:** ResNet stages downsample by changing stride.

In [ ]:
def play(stride=1, padding=1):
    H = W = 32
    kernel = 3
    out_h = (H + 2 * padding - kernel) // stride + 1
    out_w = (W + 2 * padding - kernel) // stride + 1
    print(f"formula: floor(({H} + 2*{padding} - {kernel})/{stride}) + 1 = {out_h}")
    conv = nn.Conv2d(3, 8, kernel, stride=stride, padding=padding, bias=False)
    y = conv(torch.zeros(1, 3, H, W))
    display(pd.DataFrame([{"input": (1, 3, H, W), "stride": stride, "padding": padding, "output": tuple(y.shape)}]))
    plt.figure(figsize=(3.5, 2.5))
    plt.bar(["H", "W"], [out_h, out_w], color="#79c0ff")
    plt.ylim(0, 36)
    plt.title("output size")
    plt.show()

try:
    from ipywidgets import interact, IntSlider
    interact(play,
             stride=IntSlider(value=1, min=1, max=4, step=1),
             padding=IntSlider(value=1, min=0, max=3, step=1))
except Exception:
    play()

### Step 13 — Stack blocks into the small ResNet

Now we assemble blocks into a network: a `stem` convolution, then three stages. The first block of stages 2 and 3 uses `stride=2` to halve spatial size while doubling channels. After the stages, global average pooling turns each channel map into one number.

In [ ]:
# Stack blocks into stages -> the small ResNet (or plain net if skip=False).
class SmallResNet(nn.Module):
    def __init__(self, blocks_per_stage=2, skip=True, n_classes=10):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True))

        def stage(in_ch, out_ch, stride):
            layers = [BasicBlock(in_ch, out_ch, stride, skip)]
            layers += [BasicBlock(out_ch, out_ch, 1, skip) for _ in range(blocks_per_stage - 1)]
            return nn.Sequential(*layers)

        self.stage1 = stage(16, 16, 1)
        self.stage2 = stage(16, 32, 2)             # downsample + double channels -> projection
        self.stage3 = stage(32, 64, 2)
        self.head = nn.Linear(64, n_classes)

    def forward(self, x):
        x = self.stem(x)
        x = self.stage1(x)
        x = self.stage2(x)
        x = self.stage3(x)
        x = x.mean(dim=(2, 3))                      # global average pool
        return self.head(x)

### Step 14 — Trace shapes through the whole model

Before training, send one mini-batch through the network and record each major tensor. The classifier returns `10` logits per image — one score per CIFAR-10 class.

In [ ]:
trace_net = SmallResNet(blocks_per_stage=2, skip=True).eval()
with torch.no_grad():
    x = xb_data[:4]
    t0 = trace_net.stem(x)
    t1 = trace_net.stage1(t0)
    t2 = trace_net.stage2(t1)
    t3 = trace_net.stage3(t2)
    pooled = t3.mean(dim=(2, 3))
    logits = trace_net.head(pooled)
trace_df = pd.DataFrame([
    {"step": "input", "shape": tuple(x.shape)},
    {"step": "stem", "shape": tuple(t0.shape)},
    {"step": "stage1", "shape": tuple(t1.shape)},
    {"step": "stage2", "shape": tuple(t2.shape)},
    {"step": "stage3", "shape": tuple(t3.shape)},
    {"step": "global avg pool", "shape": tuple(pooled.shape)},
    {"step": "head logits", "shape": tuple(logits.shape)},
])
display(trace_df)

### Step 15 — Parameter table + init sanity check

The ResNet and plain net have the same trainable parameter count; removing the skip does not remove convolution weights. With random weights, predictions should be near chance (`10%`) and cross-entropy should be near `ln(10)`.

In [ ]:
def param_table(net):
    groups = []
    for name, module in [("stem", net.stem), ("stage1", net.stage1), ("stage2", net.stage2), ("stage3", net.stage3), ("head", net.head)]:
        groups.append({"component": name, "params": count_params(module)})
    return pd.DataFrame(groups)

init_res = SmallResNet(blocks_per_stage=2, skip=True).to(device)
init_plain = SmallResNet(blocks_per_stage=2, skip=False).to(device)
param_df = param_table(init_res)
param_df.loc[len(param_df)] = {"component": "total", "params": int(param_df["params"].sum())}
display(param_df)
print("ResNet params:", count_params(init_res), " Plain params:", count_params(init_plain))

lf = nn.CrossEntropyLoss()
init_res.eval()
with torch.no_grad():
    logits0 = init_res(xb_data[:TEST_BATCH_SIZE].to(device))
    loss0 = lf(logits0, yb_data[:TEST_BATCH_SIZE].to(device)).item()
    acc0 = (logits0.argmax(1).cpu() == yb_data[:len(logits0)]).float().mean().item() * 100
print("random-init loss:", round(loss0, 3), " near ln(10)=", round(float(np.log(10)), 3))
print("random-init batch acc:", round(acc0, 1), "%")

**🎛️ Play with it — blocks per stage**

Choose how many residual blocks each stage contains. **Try:** `1`, `2`, and `4`. **Watch:** depth and parameter count grow without training anything. **Why it matters:** ResNet lets us make CNNs deeper while keeping them optimizable.

In [ ]:
def play(blocks_per_stage=2):
    net = SmallResNet(blocks_per_stage=blocks_per_stage, skip=True)
    conv_layers = sum(1 for m in net.modules() if isinstance(m, nn.Conv2d))
    bn_layers = sum(1 for m in net.modules() if isinstance(m, nn.BatchNorm2d))
    df = pd.DataFrame([{
        "blocks/stage": blocks_per_stage,
        "residual blocks": 3 * blocks_per_stage,
        "conv layers": conv_layers,
        "BatchNorm layers": bn_layers,
        "params": count_params(net),
    }])
    display(df)
    plt.figure(figsize=(4.5, 2.6))
    plt.bar(["conv layers", "params/1000"], [conv_layers, count_params(net) / 1000], color=["#79c0ff", "#7ee787"])
    plt.title("model size")
    plt.show()

try:
    from ipywidgets import interact, IntSlider
    interact(play, blocks_per_stage=IntSlider(value=2, min=1, max=5, step=1))
except Exception:
    play()

**🎛️ Play with it — BatchNorm on/off**

Compare the activation histogram after a convolution with and without BatchNorm. **Try:** toggling the checkbox. **Watch:** BatchNorm recenters and rescales channels before ReLU. **Why it matters:** stable activations make deeper CNN training less fragile.

In [ ]:
bn_demo_conv = nn.Conv2d(3, 16, 3, padding=1, bias=False)
bn_demo_bn = nn.BatchNorm2d(16)
with torch.no_grad():
    bn_demo_conv.weight.mul_(3.0)

def play(use_bn=True):
    with torch.no_grad():
        z = bn_demo_conv(hero_images)
        if use_bn:
            z = bn_demo_bn(z)
        a = F.relu(z)
    df = pd.DataFrame([{"BatchNorm": use_bn, "mean": float(a.mean()), "std": float(a.std()), "zero fraction": float((a == 0).float().mean())}])
    display(df.round(3))
    plt.figure(figsize=(5, 2.8))
    plt.hist(a.flatten().numpy(), bins=50, color="#ffb86b")
    plt.title("activation hist")
    plt.xlabel("activation")
    plt.ylabel("count")
    plt.show()

try:
    from ipywidgets import interact, Checkbox
    interact(play, use_bn=Checkbox(value=True, description="BatchNorm"))
except Exception:
    play()

**🎛️ Play with it — first-layer feature maps**

Pick an image and inspect a few channels after the trained-looking stem operation. **Try:** different image indices. **Watch:** channels light up on different edges, colors, or textures. **Why it matters:** CNNs turn pixels into feature maps before making class decisions.

In [ ]:
feature_stem = SmallResNet(blocks_per_stage=2, skip=True).stem.eval()

def play(image_index=0):
    image_index = int(image_index) % len(test_set)
    img, label = test_set[image_index]
    with torch.no_grad():
        fmap = feature_stem(img.unsqueeze(0))[0]
    fig, axes = plt.subplots(1, 7, figsize=(10, 1.8))
    axes[0].imshow(denorm(img).permute(1, 2, 0).numpy())
    axes[0].set_title(classes[int(label)])
    axes[0].axis("off")
    for ax, ch in zip(axes[1:], range(6)):
        ax.imshow(fmap[ch].numpy(), cmap="magma")
        ax.set_title(f"ch {ch}")
        ax.axis("off")
    plt.suptitle("first layer maps")
    plt.tight_layout()
    plt.show()

try:
    from ipywidgets import interact, IntSlider
    interact(play, image_index=IntSlider(value=0, min=0, max=min(50, len(test_set)-1), step=1))
except Exception:
    play()

### Step 16 — Accuracy and training helpers

The `accuracy` helper evaluates on the held-out test subset. The `train` function builds a fresh `SmallResNet`, optimizes it with Adam and cross-entropy, and records the per-epoch train loss and test accuracy so the results section can plot curves without retraining.

In [ ]:
def accuracy(net):
    net.eval()
    correct = total = 0
    with torch.no_grad():
        for xb, yb in test_loader:
            xb, yb = xb.to(device), yb.to(device)
            preds = net(xb).argmax(1)
            correct += (preds == yb).sum().item()
            total += yb.size(0)
    return 100.0 * correct / total


def train(skip, epochs=EPOCHS, return_model=False):
    torch.manual_seed(0)
    net = SmallResNet(blocks_per_stage=2, skip=skip).to(device)
    opt = torch.optim.Adam(net.parameters(), lr=1e-3)   # Adam (2014)
    lf = nn.CrossEntropyLoss()
    history = {"epoch": [], "train_loss": [], "test_acc": []}
    for ep in range(epochs):
        net.train()
        tot = nb = 0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            loss = lf(net(xb), yb)
            loss.backward()                              # backprop (1986)
            opt.step()
            tot += loss.item()
            nb += 1
        ep_loss = tot / nb
        ep_acc = accuracy(net)
        history["epoch"].append(ep)
        history["train_loss"].append(ep_loss)
        history["test_acc"].append(ep_acc)
        print(f"  epoch {ep}  train loss {ep_loss:.4f}  test acc {ep_acc:.1f}%")
    if return_model:
        return net, history
    return history["test_acc"][-1]

**🎛️ Play with it — learning rate note**

This sandbox does **not** retrain the model. It estimates how much one Adam step would scale with a chosen learning rate on a single batch. **Try:** `1e-4`, `1e-3`, `1e-2`. **Watch:** the estimated update size. **Why it matters:** to really change training, edit `lr=1e-3` in `train()` and rerun the ablation.

In [ ]:
lr_probe = SmallResNet(blocks_per_stage=2, skip=True).to(device)
probe_x, probe_y = next(iter(train_loader))
probe_x, probe_y = probe_x.to(device), probe_y.to(device)
probe_loss = nn.CrossEntropyLoss()(lr_probe(probe_x), probe_y)
probe_loss.backward()
grad_norm = torch.sqrt(sum((p.grad.detach() ** 2).sum() for p in lr_probe.parameters() if p.grad is not None)).item()
param_norm = torch.sqrt(sum((p.detach() ** 2).sum() for p in lr_probe.parameters())).item()
lr_probe.zero_grad(set_to_none=True)

def play(lr=1e-3):
    update_norm = lr * grad_norm
    df = pd.DataFrame([{"lr": lr, "grad norm": grad_norm, "param norm": param_norm, "rough SGD update norm": update_norm, "update/param": update_norm / param_norm}])
    display(df.round(6))
    print("To test this for real: change lr in train(), then rerun the ResNet-vs-plain cells.")
    plt.figure(figsize=(4.5, 2.7))
    plt.bar(["params", "lr*grad"], [param_norm, update_norm], color=["#79c0ff", "#ff7b72"])
    plt.yscale("log")
    plt.title("update scale")
    plt.show()

try:
    from ipywidgets import interact, FloatLogSlider
    interact(play, lr=FloatLogSlider(value=1e-3, base=10, min=-4, max=-1, step=0.25))
except Exception:
    play()

### Step 17 — Train the ResNet, then run the skip-removal ablation

Now we train the assembled ResNet (`skip=True`) and a matched plain net (`skip=False`) — same depth, width, optimizer, data, and seed, but without `+ identity`. The accuracy gap isolates the residual connection as the load-bearing idea.

Exact numbers vary by hardware and seed; this is **our small run**, not any paper's reported figure.

In [ ]:
print("ResNet (skip=True) -- the assembled classifier:")
res_model, res_history = train(skip=True, return_model=True)
res_acc = res_history["test_acc"][-1]

print("Plain net (skip=False) -- ABLATION, same depth, no residual:")
plain_model, plain_history = train(skip=False, return_model=True)
plain_acc = plain_history["test_acc"][-1]

print(f"\nFINAL  ResNet test accuracy: {res_acc:.1f}%")
print(f"FINAL  Plain  test accuracy: {plain_acc:.1f}%")
print(f"Skip connection bought us +{res_acc - plain_acc:.1f} points of test accuracy.")
# The residual net usually trains faster and scores higher than the matched plain net.
# (Exact numbers vary by hardware/seed; this is our small run, not any paper's reported number.)

**🎛️ Play with it — test image prediction**

Pick a held-out image and ask the trained ResNet for probabilities. **Try:** images it gets right and wrong. **Watch:** the probability bar, predicted class, and true class. **Why it matters:** accuracy is an aggregate; individual predictions show what the model believes.

In [ ]:
def predict_one(image_index=0):
    image_index = int(image_index) % len(test_set)
    img, label = test_set[image_index]
    res_model.eval()
    with torch.no_grad():
        logits = res_model(img.unsqueeze(0).to(device))[0]
        probs = torch.softmax(logits, dim=0).cpu()
    pred = int(probs.argmax())
    fig, axes = plt.subplots(1, 2, figsize=(8.5, 3.2))
    axes[0].imshow(denorm(img).permute(1, 2, 0).numpy())
    axes[0].set_title(f"true {classes[int(label)]}")
    axes[0].axis("off")
    colors = ["#7ee787" if i == int(label) else ("#ff7b72" if i == pred else "#79c0ff") for i in range(10)]
    axes[1].barh(classes, probs.numpy(), color=colors)
    axes[1].invert_yaxis()
    axes[1].set_xlim(0, 1)
    axes[1].set_title(f"pred {classes[pred]}")
    plt.tight_layout()
    plt.show()
    print("predicted:", classes[pred], " probability:", round(float(probs[pred]), 3), " correct:", pred == int(label))

try:
    from ipywidgets import interact, IntSlider
    interact(predict_one, image_index=IntSlider(value=0, min=0, max=min(100, len(test_set)-1), step=1))
except Exception:
    predict_one()

## Visualize the data & results

_Does our assembled small ResNet train (accuracy up, loss down), and does removing the skip hurt?_

### Step 18 — Accuracy and loss curves

Because `train()` recorded each epoch, we can plot the ablation directly. The two models saw the same images in the same order; the only switch is the residual connection.

In [ ]:
history_df = pd.concat([
    pd.DataFrame(res_history).assign(model="ResNet"),
    pd.DataFrame(plain_history).assign(model="Plain"),
], ignore_index=True)
display(history_df.round(3))

fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
for name, hist, color in [("ResNet", res_history, "#7ee787"), ("Plain", plain_history, "#ff7b72")]:
    axes[0].plot(hist["epoch"], hist["train_loss"], marker="o", label=name, color=color)
    axes[1].plot(hist["epoch"], hist["test_acc"], marker="o", label=name, color=color)
axes[0].set_title("train loss")
axes[0].set_xlabel("epoch")
axes[0].set_ylabel("loss")
axes[1].set_title("test accuracy")
axes[1].set_xlabel("epoch")
axes[1].set_ylabel("accuracy (%)")
for ax in axes:
    ax.legend()
plt.tight_layout()
plt.show()

### Step 19 — Final results table and bar chart

This is the controlled comparison in one row per model. Parameter counts are equal; the measured difference is the skip connection.

In [ ]:
results_df = pd.DataFrame([
    {"model": "ResNet", "skip": True, "params": count_params(res_model), "final test acc": res_acc, "final train loss": res_history["train_loss"][-1]},
    {"model": "Plain", "skip": False, "params": count_params(plain_model), "final test acc": plain_acc, "final train loss": plain_history["train_loss"][-1]},
])
display(results_df.round(3))

plt.figure(figsize=(5.5, 3.3))
plt.bar(results_df["model"], results_df["final test acc"], color=["#7ee787", "#ff7b72"])
plt.ylabel("test accuracy (%)")
plt.title("final accuracy")
plt.ylim(0, max(100, float(results_df["final test acc"].max()) + 5))
plt.show()

### Step 20 — Confusion matrix on the test subset

A confusion matrix shows *which* classes are being mixed up. Rows are true classes; columns are predicted classes. A strong model has a bright diagonal.

In [ ]:
def collect_predictions(net):
    net.eval()
    ys, ps = [], []
    with torch.no_grad():
        for xb, yb in test_loader:
            logits = net(xb.to(device))
            ps.append(logits.argmax(1).cpu())
            ys.append(yb)
    return torch.cat(ys), torch.cat(ps)

true_y, pred_y = collect_predictions(res_model)
cm = torch.zeros(10, 10, dtype=torch.int64)
for t, p in zip(true_y, pred_y):
    cm[int(t), int(p)] += 1
cm_df = pd.DataFrame(cm.numpy(), index=classes, columns=classes)
display(cm_df)

plt.figure(figsize=(6.5, 5.5))
plt.imshow(cm.numpy(), cmap="Blues")
plt.xticks(range(10), classes, rotation=45, ha="right")
plt.yticks(range(10), classes)
plt.xlabel("predicted")
plt.ylabel("true")
plt.title("confusion matrix")
plt.colorbar(fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()

### Step 21 — Sample predictions: green correct, red wrong

The grid below makes the confusion matrix concrete. Titles are green for correct predictions and red for mistakes.

In [ ]:
res_model.eval()
fig, axes = plt.subplots(3, 4, figsize=(8, 6))
with torch.no_grad():
    for ax, idx in zip(axes.ravel(), range(12)):
        img, label = test_set[idx]
        probs = torch.softmax(res_model(img.unsqueeze(0).to(device))[0], dim=0).cpu()
        pred = int(probs.argmax())
        ax.imshow(denorm(img).permute(1, 2, 0).numpy())
        color = "green" if pred == int(label) else "red"
        ax.set_title(f"{classes[int(label)]}->{classes[pred]}", color=color)
        ax.axis("off")
plt.suptitle("sample predictions")
plt.tight_layout()
plt.show()

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The capstone ablation. You have just trained the assembled small ResNet and printed its
            CIFAR-10 test accuracy. Now set skip=False to delete the single
            "+ identity" line in every block (a matched plain net of identical depth, width,
            optimizer, and data) and retrain. What happens to test accuracy, and what does the
            comparison prove about which paper's idea is load-bearing?

In [ ]:
# Your turn:

<details><summary>Show worked solution</summary>

- Change exactly one thing: the block's skip flag from True to False, so out = relu(out + identity) becomes out = relu(out). Keep depth, width, BatchNorm, Adam, seed, and the CIFAR-10 subset identical. — _An honest ablation varies one factor &mdash; the residual connection &mdash; so any accuracy gap is attributable to it alone, not to capacity or tuning._
- Retrain and compare: the residual net's test accuracy is clearly higher; the deep plain net trains more slowly and lands lower (or stalls). — _Without the "$+x$" gradient highway, the signal to early layers is weak, so the deep plain stack optimizes poorly &mdash; the degradation problem reproduced on our small run._
- Conclude that the skip connection, not extra parameters, is what makes the deep classifier work. — _Both nets share layers and parameter count; only the residual one trains well, isolating ResNet's contribution from all the others._

**Answer:** The residual net reaches clearly higher CIFAR-10 test accuracy than the matched
                 plain net, which trains slowly and plateaus low. Because the two are identical
                 except for the "+ identity", this isolates ResNet's skip connection as
                 the load-bearing idea for depth &mdash; an optimization fix, not a parameter-count
                 or regularization effect. The CODEVIZ panel shows the same gap in the training curves.
                 (These are our small-run numbers, not any paper's reported result.)

</details>

**Problem 2.** Trace one component to one line of the final build. For each of (a) convolution, (b) Batch
            Normalization, (c) the residual skip, (d) the optimizer, name the paper that introduced it and
            the PyTorch construct in our code that embodies it.

In [ ]:
# Your turn:

<details><summary>Show worked solution</summary>

- Map convolution &rarr; LeNet &rarr; nn.Conv2d(...) (the 3&times;3 layers, with VGG's stacking recipe). — _LeNet introduced weight-shared sliding filters; VGG fixed the size at 3&times;3 and stacked them._
- Map normalization &rarr; BatchNorm &rarr; nn.BatchNorm2d(...) after each conv. — _Standardize-then-scale per channel over the mini-batch is exactly the BatchNorm paper._
- Map the skip &rarr; ResNet &rarr; out = relu(out + identity); and the step rule &rarr; Adam &rarr; torch.optim.Adam(...). ReLU traces to AlexNet, and loss.backward() to backprop. — _Each line of the build is one paper's contribution made concrete._

**Answer:** (a) Convolution &rarr; LeNet &rarr; nn.Conv2d (3&times;3, stacked per
                 VGG). (b) Batch Normalization &rarr; BatchNorm &rarr; nn.BatchNorm2d.
                 (c) The residual skip &rarr; ResNet &rarr; out = relu(out + identity).
                 (d) The optimizer &rarr; Adam &rarr; torch.optim.Adam. (ReLU is
                 AlexNet's; the backward sweep loss.backward() is backprop.) The
                 whole network is these seven papers assembled.

</details>